# Movie Recommendation System (Content-Based)

***Performed by:*** <br>
Name: Giridhar Sreekumar </br>
Date : 20 January 2026 </br>

## Description

Recommend movies to users based on movie descriptions using ***content-based filtering***. The system identifies movies with similar textual content using ***TF-IDF vectorization*** and ***cosine similarity.***

In [49]:
##All required libraries are imported
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [50]:
##Dataset loading
df=pd.read_csv('/content/movies.csv')
df = df[['original_title', 'overview', 'genres', 'keywords', 'cast', 'director']]
df.head()


,original_title,overview,genres,keywords,cast,director
0,Jurassic World,Twenty-two years after the events of Jurassic ...,Action|Adventure|Science Fiction|Thriller,monster|dna|tyrannosaurus rex|velociraptor|island,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,Colin Trevorrow
1,Mad Max: Fury Road,An apocalyptic story set in the furthest reach...,Action|Adventure|Science Fiction|Thriller,future|chase|post-apocalyptic|dystopia|australia,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,George Miller
2,Insurgent,Beatrice Prior must confront her inner demons ...,Adventure|Science Fiction|Thriller,based on novel|revolution|dystopia|sequel|dyst...,Shailene Woodley|Theo James|Kate Winslet|Ansel...,Robert Schwentke
3,Star Wars: The Force Awakens,Thirty years after defeating the Galactic Empi...,Action|Adventure|Science Fiction|Fantasy,android|spaceship|jedi|space opera|3d,Harrison Ford|Mark Hamill|Carrie Fisher|Adam D...,J.J. Abrams
4,Furious 7,Deckard Shaw seeks revenge against Dominic Tor...,Action|Crime|Thriller,car race|speed|revenge|suspense|car,Vin Diesel|Paul Walker|Jason Statham|Michelle ...,James Wan


##Initial Inspection

Initial inspection is performed to gain a basic understanding of the dataset structure, number of records, feature types, and data completeness before text preprocessing.

In [51]:
df.shape

(10866, 6)

In [52]:
df.columns

Index(['original_title', 'overview', 'genres', 'keywords', 'cast', 'director'], dtype='object')

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10866 entries, 0 to 10865
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   original_title  10866 non-null  object
 1   overview        10862 non-null  object
 2   genres          10843 non-null  object
 3   keywords        9373 non-null   object
 4   cast            10790 non-null  object
 5   director        10822 non-null  object
dtypes: object(6)
memory usage: 509.5+ KB


In [54]:
##Checking missing values
df.isnull().sum()

,0
original_title,0
overview,4
genres,23
keywords,1493
cast,76
director,44


###**Missing Value Insight**

Some text-based columns such as overview, keywords, cast, and tagline contain missing values. These missing values are handled by replacing them with empty strings to avoid issues during text combination and TF-IDF vectorization.

In [55]:
##Handle duplicates
df.duplicated().sum() ##Initially came out to be 1
df=df.drop_duplicates().reset_index(drop=True)
df.duplicated().sum() ##Now came out to be 0

np.int64(0)

***Inference*** : When checked initially, there was presence of one duplicate record. Since the duplication was minimal, it was easily handled with the help of ***drop_duplicates()*** function and no of duplicates were checked which came out to be 0 which proved that duplicates has been removed.

In [56]:
## Data Cleaning
df.fillna('',inplace=True)

##**EDA Conclusion**

* The initial inspection confirmed that the TMDB dataset is rich in textual metadata required for building a content-based recommendation system.
* While traditional numerical analysis and visualizations are not applicable, careful inspection and preparation of text features are crucial.
* After handling missing values, the dataset is suitable for text vectorization and similarity computation.

## Feature Engineering
Since this is a content-based recommender, relevant textual features are combined into single column to represent each movie's content.

***Selected features:***
* overview
* genres
* keywords
* cast
* director

In [57]:
###Combine text feature
df['combined_features'] = (
    df['overview'] + ' ' +
    df['genres'] + ' ' +
    df['keywords'] + ' ' +
    df['cast'] + ' ' +
    df['director']
)

**Note:** Combining multiple text features improves recommendation quality by capturing rich movie content.


## Text Vectorization - Inverse Document Frequency (TF-IDF)
* TF-IDF helps to convert text data into numeric features while reducing the impact of common words.

In [58]:
tfidf= TfidfVectorizer(stop_words='english')
tfidf_matrix=tfidf.fit_transform(df['combined_features'])

**Reasoning:**
* Converts textual movie data into numerical vectors.
* Gives higher weight to important and unique words.(Ex. word ***the*** appears in almost every movie but words like ***Dream*** are frequent for movies like Inception)
* Reduces the impact of commonly occuring words.
* Helps represent movie content in a meaningul way.

***In the matrix, we get:***

- A huge matrix of numbers.
- Each row = one movie
- Each column = one word
- Each values = how important that word is to that movie.

So every movie is obtained in a numeric vector form.




## Similarity Computation (Cosine Similarity)
* Cosine Similarity measures how similar two movies are based on TF-IDF vectors.

In [59]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

***Reasoning:***
 Helps us answer the question how similar are two number vectors.

 **Simple Intuition:**
 * It checks if two movies talk about similar things.
 * It doesn't check how long a description is or what is means, just that if the movie pose a resemblance with the other.
 * So, basically, it checks if the direction of meaning of the description is similar
 * Helpful in creating movie recommendations.
<hr>

` 1 ` → very similar

`0` → not similar

`-1` → opposite (rare in text)





In [60]:
## Index Mapping
### This mapping allows quick lookup of movie indices using movie titles.

indices= pd.Series(df.index, index=df['original_title']).drop_duplicates()

🎯 ***Recommendation Function***

* This function takes an input from the user and with the help of the predefined methods help us create a recommendation system that returns 5 similar movies back to the user.

In [61]:
def recommend_movies(title, num_recommendations = 5):
  idx=indices[title]

  sim_scores=list(enumerate(cosine_sim[idx]))
  sim_scores=sorted(sim_scores, key=lambda x : x[1],reverse = True)

  sim_scores = sim_scores[1:num_recommendations + 1]
  movie_indices = [i[0] for i in sim_scores]

  return df['original_title'].iloc[movie_indices]

** Reasoning: **
The recommendation function is designed to identify and return movies that are most similar in content to a given input movie. It works by first locating the index of the selected movie in the dataset and then retrieving its similarity scores with all the other movies using cosine similarity matrix. These scores are sorted in descending order to find movies with the highest similarity. The function excludes the input movie itself and returns the top recommendation based on content similarity.

In [62]:
## Sample Recommendation 1
recommend_movies("Inception")

,original_title
2184,Inception: The Cobol Job
1395,(500) Days of Summer
5459,Don Jon
2095,Hesher
5,The Revenant


In [63]:
##Sample Recommendation 2
recommend_movies("The Dark Knight")


,original_title
4362,The Dark Knight Rises
6190,Batman Begins
8244,Batman Returns
8081,Batman Forever
3245,Batman Unmasked: The Psychology of the Dark Kn...


In [64]:
##Sample Recommendation 3
recommend_movies("Avatar")

,original_title
2591,Babylon 5: A Call to Arms
6119,Lifeforce
630,Guardians of the Galaxy
5233,Gattaca
7828,Moonraker


In [65]:
recommend_movies("(500) Days of Summer")

,original_title
5459,Don Jon
3949,The Good Girl
3008,The Happening
3474,Your Highness
2582,Mumford


In [67]:
recommend_movies("The Matrix")

,original_title
4952,The Matrix Revolutions
4953,The Matrix Reloaded
2766,The Matrix Revisited
11,Jupiter Ascending
4383,Cloud Atlas


**Limitation Note ( Title Duplication)**

Some movie titles appear multiple times in dataset, which can lead to ambiguous similarity lookups for certain inputs. As a result, a small number of recommendations may not perfectly align with thematic expectations. This behavior reflects a known limiation of basic content-based recommendation systems that rely on textual similarity and title-based indexing rather than unique identifiers.

 📊 ***<u>Model Evaluation Summary</u>***
- This content-based recommendation system does not use traiditonal evaluation metrics such as accuracy or confusion matrix.
- The model's effectiveness is assessed qualitatively by verifying that recommended movies share similar themes, genres, cast, or direction with the input movies.

**================== CONCLUSION ========================**

* A content-based movie recommendation system was successfully developed using the TMDB dataset.

* Relevant textual features were combined and transformed using TF-IDF vectorization.

* Cosine similarity was used to identify movies with similar content.

* The system provide meaningful movie recommendations without relying on user ratings or interaction history.

* This project demonstrates the practical application of NLP and similarity-based machine learning techniques in recommendation systems.
